%% [markdown]
# ESM-2 training parity — CPU / CUDA / Trainium  (M5: on-device training)
The third parity check (after 01-inference, 02-backprop): a real **multi-step training loop** — forward →
CrossEntropy loss → backward → **`optimizer.step()`** — on a composite (ESM-2 backbone + a new head), run
on every device, checking the **loss trajectory** and **final weights** match CPU. This proves that
`optimizer.step()`, the Adam optimizer state, and multi-step convergence all work on the Trainium core —
i.e. you can actually *train* new models on these backbones, not just run one backward.

`cuda` auto-skips when absent. Pin a free core:
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 03_training_parity.ipynb`.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import esm2_reference as R

[W709 05:20:38.305974521 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())


In [2]:
K = 8               # training steps
LR = 1e-3
LABELS = torch.tensor([0, 1])       # deterministic binary labels for the fixed batch

In [3]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [4]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES, "| model:", R.MODEL_NAME, f"| {K} steps, Adam lr={LR}")

torch 2.11.0+cpu | devices: ['cpu', 'neuron'] | model: facebook/esm2_t6_8M_UR50D | 8 steps, Adam lr=0.001


In [5]:
# %% [markdown]
# ## Train K steps on each device (identical seed / init / batch / optimizer)
# %%
def train(device):
    torch.manual_seed(0)                                  # identical init across devices
    model = R.load(device, finetune_backbone=True)        # ESM-2 backbone + new head, ALL params trainable
    ids, mask = R.build_inputs()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    losses = []
    for _ in range(K):
        opt.zero_grad(set_to_none=True)
        _, logits = model(ids.to(device), mask.to(device))
        loss = F.cross_entropy(logits, LABELS.to(device))
        loss.backward()
        opt.step()
        if device == "neuron":
            import torch_neuronx; torch_neuronx.synchronize()
        losses.append(float(loss.detach().cpu()))
    weights = {n: p.detach().float().cpu() for n, p in model.named_parameters()}
    return losses, weights

In [6]:
results = {d: train(d) for d in DEVICES}
for d in DEVICES:
    print(f"{d:7s} loss: {[round(x, 4) for x in results[d][0]]}")
lc = results["cpu"][0]
print(f"\nloss decreased on CPU: {lc[0]:.4f} -> {lc[-1]:.4f}  ({'learning' if lc[-1] < lc[0] else 'NO decrease'})")

/home/user/miniconda3/envs/torch-neuron/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 17021.52it/s]


[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 13799.72it/s]


[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Neuron backend does not support int64. Automatically casting to int32. Consider using int32 directly for better performance until native int64 support is added.


cpu     loss: [0.6854, 0.8756, 0.6737, 0.3548, 0.1049, 0.0482, 0.0271, 0.0161]
neuron  loss: [0.6854, 0.8756, 0.6737, 0.3548, 0.1049, 0.0482, 0.0271, 0.0161]

loss decreased on CPU: 0.6854 -> 0.0161  (learning)


In [7]:
# %% [markdown]
# ## Check the training trajectory + final weights match CPU
# %%
ref_loss, ref_w = results["cpu"]
all_ok = lc[-1] < lc[0]        # must actually learn
for d in DEVICES:
    if d == "cpu":
        continue
    dl, dw = results[d]
    max_loss_diff = max(abs(a - b) for a, b in zip(ref_loss, dl))
    max_w_diff = max((ref_w[n] - dw[n]).abs().max().item() for n in ref_w)
    ok = max_loss_diff < 1e-2 and max_w_diff < 1e-2
    all_ok = all_ok and ok
    print(f"{d} vs cpu: max per-step loss diff={max_loss_diff:.3e}  max final-weight diff={max_w_diff:.3e}  {'OK' if ok else 'FAIL'}")

neuron vs cpu: max per-step loss diff=4.053e-06  max final-weight diff=1.254e-03  OK


In [8]:
print("\nTRAINING PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "training trajectory / final weights diverged across devices, or the loss did not decrease"


TRAINING PARITY: PASS
